<a href="https://colab.research.google.com/github/Sushantak17/MLOps-Sushantak-M25CSA035/blob/Assignment_1/Assignment1_MNIST_FashionMNIST.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split
import matplotlib.pyplot as plt


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

USE_AMP = True
scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)


In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

mnist_full = datasets.MNIST(
    root="./data", train=True, download=True, transform=transform
)

fashion_full = datasets.FashionMNIST(
    root="./data", train=True, download=True, transform=transform
)

def split_dataset(dataset):
    total = len(dataset)
    train_size = int(0.7 * total)
    val_size = int(0.1 * total)
    test_size = total - train_size - val_size
    return random_split(dataset, [train_size, val_size, test_size])

mnist_train, mnist_val, mnist_test = split_dataset(mnist_full)
fashion_train, fashion_val, fashion_test = split_dataset(fashion_full)

print("MNIST:", len(mnist_train), len(mnist_val), len(mnist_test))
print("FashionMNIST:", len(fashion_train), len(fashion_val), len(fashion_test))


In [ ]:
def get_dataloaders(train_ds, val_ds, test_ds, batch_size, pin_memory):
    train_loader = DataLoader(
        train_ds, batch_size=batch_size, shuffle=True, pin_memory=pin_memory
    )
    val_loader = DataLoader(
        val_ds, batch_size=batch_size, shuffle=False, pin_memory=pin_memory
    )
    test_loader = DataLoader(
        test_ds, batch_size=batch_size, shuffle=False, pin_memory=pin_memory
    )
    return train_loader, val_loader, test_loader


In [ ]:
def get_resnet(model_name, num_classes=10):
    if model_name == "resnet18":
        model = models.resnet18(weights=None)
    elif model_name == "resnet50":
        model = models.resnet50(weights=None)
    else:
        raise ValueError("Invalid model name")

    # Modify first conv layer (MNIST is 1-channel)
    model.conv1 = nn.Conv2d(
        1, 64, kernel_size=7, stride=2, padding=3, bias=False
    )

    # Modify final classification layer
    model.fc = nn.Linear(model.fc.in_features, num_classes)

    return model.to(device)


In [ ]:
criterion = nn.CrossEntropyLoss()


In [ ]:
def train_one_epoch(model, loader, optimizer):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()

        with torch.cuda.amp.autocast(enabled=USE_AMP):
            outputs = model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    return running_loss / len(loader), 100 * correct / total


def evaluate(model, loader):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

    return 100 * correct / total


In [ ]:
# Example experiment
batch_size = 16
pin_memory = True
epochs = 5

train_loader, val_loader, test_loader = get_dataloaders(
    mnist_train, mnist_val, mnist_test, batch_size, pin_memory
)

model = get_resnet("resnet18")

optimizer = optim.Adam(model.parameters(), lr=0.001)

for epoch in range(epochs):
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer)
    val_acc = evaluate(model, val_loader)

    print(f"Epoch {epoch+1}/{epochs} | "
          f"Loss: {train_loss:.4f} | "
          f"Train Acc: {train_acc:.2f}% | "
          f"Val Acc: {val_acc:.2f}%")

test_acc = evaluate(model, test_loader)
print("Test Accuracy:", test_acc)
